# 第 21 章 · 后台宿主：连接离开，任务属于谁

终端断开是观察者离开；任务取消是控制命令。把二者写成同一个 `connected` 状态，会让断线错误地终止工作，或让取消永远停不下来。

本章补充一个**同进程、本地文件 journal 的教学宿主**。它确实读写磁盘、校验连接与任务状态，但没有后台进程、RPC、配对密码学、真实 worker 或 OS 隔离。四个 owner 分开：host 接纳任务、run 保存进度、trust 保存控制者代次、session 保存临时连接。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 幂等提交和断线

服务接纳请求后，任务身份由宿主持有。`key` 绑定原指令；同键不同内容必须报错。改变 `instruction`，观察冲突在创建第二个任务之前被拒绝。

In [2]:
m = load_example('21', 'local_host')
run_dir = fresh_run_dir('ch21')
host = m.LocalHost(run_dir)
generation = host.pair('reader')
session = host.connect('reader', generation)
instruction = '修复 task-list 的 status 筛选'
run_id = host.submit(session, 'request-1', instruction)
assert host.submit(session, 'request-1', instruction) == run_id
try:
    host.submit(session, 'request-1', instruction + '，另建任务')
except m.Refused as error:
    assert str(error) == 'idempotency_key_conflict'
else:
    raise AssertionError('同键不同指令应拒绝')
host.start(run_id)
host.disconnect(session)
assert host.runs[run_id]['state'] == 'running'
print({'state': host.runs[run_id]['state'], 'connections': len(host.sessions), 'journal': str(host.path)})

{'state': 'running', 'connections': 0, 'journal': '<project>/runs/notebooks/ch21-20261113T124122Z-7aa0bbbc/journal.jsonl'}


## 2 · 重连读游标，身份仍要经过动作策略

重连从 journal 游标补读，不重复提交。认证仅说明是谁；能否修改文件仍由动作策略判断。将 `allowed` 改为 True 后，策略拒绝断言应失败，这说明它在检查实际分支。

In [3]:
session2 = host.connect('reader', generation)
events = host.read_after(session2, 0)
assert len(events) >= 3
cursor = events[-1]['cursor']
assert host.read_after(session2, cursor) == []
try:
    host.authorize(session2, allowed=False)
except m.Refused as error:
    assert str(error) == 'action_policy_denied'
else:
    raise AssertionError('合法连接不应绕过动作权限')
host.revoke('reader')
try:
    host.read_after(session2, cursor)
except m.Refused as error:
    assert str(error) == 'trust_revoked'
else:
    raise AssertionError('撤销后的旧连接必须拒绝')
print({'cursor': cursor, 'task_after_revoke': host.runs[run_id]['state']})

{'cursor': 3, 'task_after_revoke': 'running'}


## 3 · 取消请求与清理确认是两步

取消不能删除已发生的事实。这里没有真实 worker，因此 `cleanup_confirmed=True` 是实验显式提供的受控输入，只验证状态转移。生产接回时，这个布尔值必须来自执行器清理回执。

In [4]:
new_generation = host.pair('reader')
session3 = host.connect('reader', new_generation)
host.cancel(session3, run_id)
assert host.runs[run_id]['state'] == 'cancel_requested'
try:
    host.acknowledge_cancel(run_id, cleanup_confirmed=False)
except m.Refused as error:
    assert str(error) == 'cleanup_not_confirmed'
else:
    raise AssertionError('未确认清理不能完成取消')
host.acknowledge_cancel(run_id, cleanup_confirmed=True)
assert host.runs[run_id]['state'] == 'cancelled'
print(host.runs[run_id])

{'state': 'cancelled', 'instruction': '修复 task-list 的 status 筛选'}


## 4 · drain 封住入口，重建不复活句柄

排队任务还没有开始副作用；running 任务在恢复时结果未知，应进入核对态。新建同一 journal 的对象模拟重建，不模拟杀进程。把 `host.start(uncertain)` 注释掉，恢复状态应变为 queued，而非 needs_reconciliation。

In [5]:
queued = host.submit(session3, 'request-2', '等待下一次调度')
uncertain = host.submit(session3, 'request-3', '可能已经写文件')
host.start(uncertain)
drain = host.drain()
assert drain['queued'] == 1 and uncertain in drain['unsettled']
try:
    host.submit(session3, 'request-4', 'drain 之后的新请求')
except m.Refused as error:
    assert str(error) == 'host_draining'
else:
    raise AssertionError('drain 后不应接纳')
restored = m.LocalHost(run_dir)
assert not restored.sessions
assert restored.runs[queued]['state'] == 'queued'
assert restored.runs[uncertain]['state'] == 'needs_reconciliation'
assert restored.runs[run_id]['state'] == 'cancelled'
print({'drain': drain, 'restored_states': {k: v['state'] for k, v in restored.runs.items()}})

{'drain': {'queued': 1, 'unsettled': ['f979709813bb4bb9a22f6c1cb75bd89c']}, 'restored_states': {'76aa3bdee47e4cce95d98b06b2b54a6a': 'cancelled', '190f963e7846459fb96a331fea6000ae': 'queued', 'f979709813bb4bb9a22f6c1cb75bd89c': 'needs_reconciliation'}}


本实验没有证明 daemon 脱离终端运行。真正的宿主还需要进程身份核对、启动互斥、RPC readiness、日志落盘顺序和 OS 层取消证据；原作者第21章合同中的这些义务保持待验。

## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/app-server-daemon/src/lib.rs#L425-L505](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/app-server-daemon/src/lib.rs#L425-L505)
- [codex-rs/app-server-daemon/src/lib.rs#L98-L111](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/app-server-daemon/src/lib.rs#L98-L111)
- [codex-rs/app-server-daemon/src/backend/pid_start.rs#L22-L91](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/app-server-daemon/src/backend/pid_start.rs#L22-L91)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。